# VLM-DENTAL: Speculative Decoding + Serving Benchmark

Run **after** a trained with-tools checkpoint is saved, in a fresh serving runtime. This notebook merges into a new local BF16 checkpoint, verifies MTP tensor provenance, benchmarks dynamic dev episodes, optionally trains only the MTP head, and compares acceptance before/after. No test split, frozen-protocol change, SFT or GRPO launch. Hardware measurements remain pending until these cells execute.

## 1. Isolated checkout
Apply the delivered harness patches to your repository and push them yourself first. Set REPO_REF to your reviewed harness commit. This creates a separate checkout; it does not pull over running training files.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"
REPO_REF = "main"  # or the exact reviewed commit containing this patch
if not Path("dental_agent/harness/notebook_eval.py").is_file():
    parent = Path("/kaggle/working" if Path("/kaggle").exists() else "/content" if Path("/content").exists() else Path.cwd())
    work = parent / "VLM-DENTAL-EVALUATION"
    if not work.exists():
        if len(REPO_REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REPO_REF):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(work)], check=True)
            subprocess.run(["git", "-C", str(work), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
            subprocess.run(["git", "-C", str(work), "checkout", "--detach", "FETCH_HEAD"], check=True)
        else:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(work)], check=True)
    os.chdir(work)
assert Path("dental_agent/harness/notebook_eval.py").is_file(), "Use the checkout containing this patch"
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
print("Checkout:", Path.cwd())

## 2. Dependencies and credentials
Prefer a fresh notebook runtime. Existing training environments are not modified. Enable installation here only in this isolated runtime. Dependency versions remain those declared by the repository.

In [ ]:
INSTALL_DEPENDENCIES = True  # install only in this isolated evaluation runtime
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[api,dev]"], check=True)
from dotenv import load_dotenv
load_dotenv()
for key in ("HF_TOKEN", "HF_ARTIFACT_REPO", "DENTEX_IMAGES_REPO"):
    if os.environ.get(key): continue
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(key)
        elif IS_COLAB:
            from google.colab import userdata
            value = userdata.get(key)
        else: value = None
        if value: os.environ[key] = value
    except Exception: pass
print("Artifacts:", os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models"))

## 3. Profile, checkpoint and fixed dev inputs
Hardware profile determines experiment identity and numerical runtime settings. Never silently substitute another checkpoint. A GRPO checkpoint should point at its step/final folder; its grpo_policy adapter is selected by the existing evaluator. No optimizer state is needed for inference.

In [ ]:
from dental_agent.harness.notebook_eval import NotebookEvaluations
import json
# Profiles are visible here. Aliases: rtx4090 -> rtx4090_qlora; rtxpro6000_48gb -> workstation48_lora.
EVAL_PROFILES = {
    "dual_t4": {
        "tag": "dual_t4_fp16",
        "precision": "qlora",
        "compute": "fp16",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    },
    "rtx4090_qlora": {
        "tag": "rtx4090_bf16",
        "precision": "qlora",
        "compute": "bf16",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "workstation48_lora": {
        "tag": "workstation48_lora_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "tpu": {
        "tag": "tpu_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "off",
        "loader": "standard",
        "threads": 0,
        "letterbox": True
    },
    "dual_t4_fp16": {
        "tag": "dual_t4_unquantized_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    }
}
# Existing result repository; checkpoints still come from HF_ARTIFACT_REPO.
EVAL_RESULTS_REPO = "Reza-Nadimi/vlm-dental-evaluations"
EVAL_RESULTS_REPO_TYPE = None  # Detect the existing model or dataset repository; never create one.
EVAL_TRACK = "with_tools"  # with_tools, no_tools, or all
print(json.dumps(EVAL_PROFILES, indent=2))
EVAL_TARGET_PROFILES = "workstation48_lora"  # any key above, a list of keys, or "all"

HARDWARE_PROFILE = "workstation48_lora"  # serving runtime for optional MTP training below
from dental_agent.harness.notebook_eval import select_profiles
HARDWARE_PROFILE = select_profiles(HARDWARE_PROFILE, EVAL_PROFILES)[0]
PROFILE = EVAL_PROFILES[HARDWARE_PROFILE]
EVAL_STAGES = "all"
EVAL_SELECTED_K = 4
DATA_DIR = "data"
DEV_SPLIT = "data/splits/dentex_dev.json"
MODEL_ID = None
SEED = 42
MAX_NEW_TOKENS = 16384
MAX_TOOL_CALLS = 50
MAX_TURNS = None
LIMIT = None
SPEC_CHECKPOINTS = {}  # {profile: {exact Hub adapter prefix: audited merged checkpoint folder}}
RUN_EVALUATIONS = True

In [ ]:
spec_evaluations = NotebookEvaluations(track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES,
    kinds=["speculative"], stages=EVAL_STAGES, selected_k=EVAL_SELECTED_K,
    model_id=MODEL_ID, data_dir=DATA_DIR, dev_split=DEV_SPLIT, split="dev", spec_checkpoints=SPEC_CHECKPOINTS)
spec_evaluations.list()

## 4. Separate serving and Speculators environments
Do not install vLLM into an active training environment. Use separate virtual environments; the repository dependencies remain unchanged. Version pins below are explicit candidates, not a claim that this hardware/model combination was verified. The environment probe records actual versions and CLI support. Existing environments can be supplied instead.

In [ ]:
VLLM_VERSION = '0.31.0'
SPECULATORS_VERSION = '0.8.1'
VLLM_ENV = Path('data/serving_env')
SPECULATORS_ENV = Path('data/speculators_env')
INSTALL_SERVING = False
if INSTALL_SERVING:
    for folder,package in [(VLLM_ENV,f'vllm=={VLLM_VERSION}'),(SPECULATORS_ENV,f'speculators=={SPECULATORS_VERSION}')]:
        if not folder.exists():subprocess.run([sys.executable,'-m','venv',str(folder)],check=True)
        py = folder / ('Scripts/python.exe' if os.name=='nt' else 'bin/python')
        subprocess.run([str(py),'-m','pip','install',package],check=True)
VLLM = str((VLLM_ENV / ('Scripts/vllm.exe' if os.name=='nt' else 'bin/vllm')).resolve())
SPECULATORS = str((SPECULATORS_ENV / ('Scripts/speculators.exe' if os.name=='nt' else 'bin/speculators')).resolve())
SPECULATORS_PYTHON = str((SPECULATORS_ENV / ('Scripts/python.exe' if os.name=='nt' else 'bin/python')).resolve())
GPU_INDICES = '0,1' if HARDWARE_PROFILE.startswith('dual_t4') else '0'
TENSOR_PARALLEL = 2 if HARDWARE_PROFILE.startswith('dual_t4') else 1
SERVING_DTYPE = 'float16' if HARDWARE_PROFILE.startswith('dual_t4') else 'bfloat16'
MAX_MODEL_LEN = 32768  # total serving capacity; generation remains 16384  # Never silently lower context to fit a smaller GPU.
GPU_MEMORY_UTILIZATION = 0.90
PORT = 8910

## 5. Read-only environment / memory report
No weights are loaded here. About 18 GiB of checkpoint tensors is a lower bound, not a 24 GB fit guarantee. Runtime failure, unsupported flags, or missing metrics are reported rather than hidden.

In [ ]:
ENV_REPORT = "data/evaluations/specdec_environment.json"
subprocess.run([sys.executable,"scripts/bench/probe_environment.py","--vllm-executable",VLLM,
                "--gpu-indices",GPU_INDICES,"--output",ENV_REPORT],check=True)
print(Path(ENV_REPORT).read_text())

## 6. Merge and preserve the native MTP head
The model loads in BF16 on CPU; allow enough host RAM and disk. The output directory must be new. For GRPO, merge only its trained grpo_policy adapter, never reference. The base checkpoint and adapter remain untouched. Merging from the original 16-bit base is a serving variant of QLoRA, so equivalence is checked against this merged baseline, not assumed against prior 4-bit inference.

In [ ]:
from dental_agent.harness.notebook_eval import prepare_serving_targets
RUN_MERGE = False  # explicit opt-in CPU merge; originals are never changed
spec_evaluations.vllm = VLLM
spec_evaluations.gpu_indices = GPU_INDICES
spec_evaluations.port = PORT
spec_evaluations.serving_context_length = MAX_MODEL_LEN
SPEC_CHECKPOINTS = prepare_serving_targets(spec_evaluations, merge=RUN_MERGE)
print(json.dumps(SPEC_CHECKPOINTS, indent=2))
# Choose an exact adapter prefix only if opting into MTP training below.
MTP_PROFILE = HARDWARE_PROFILE
MTP_ADAPTER_PREFIX = None
MERGED = SPEC_CHECKPOINTS.get(MTP_PROFILE, {}).get(MTP_ADAPTER_PREFIX)
if RUN_MERGE: MODEL_ID = spec_evaluations.model_id

## 7. Fixed dev selection and exact commands
All conditions use the same merged target, full system/tool prompt, seed, views and budgets. The benchmark starts one exclusive server per condition and stops only processes it owns. It refuses an occupied port. Warmup episodes are identical and excluded from timing.

In [ ]:
BENCH_CONDITIONS = ["baseline", "mtp_1"]  # one same-target speculative comparison
# Every target/profile gets a separate identity; no merged target is reused for a different adapter.
print("Selected profiles:", spec_evaluations.profiles)

In [ ]:
SPEC_EVAL_STATUS = spec_evaluations.run(execute=RUN_EVALUATIONS)
print(json.dumps(SPEC_EVAL_STATUS, indent=2))

## 8. Accuracy, speed and acceptance
Inspect the recorded equivalence gate alongside pooled exact F1, latency, throughput and measured acceptance. A divergent benchmark remains failed evidence.

In [ ]:
from dental_agent.harness.notebook_eval import summarize_inventory
SPEC_SUMMARY = summarize_inventory(SPEC_EVAL_STATUS)
print(json.dumps(SPEC_SUMMARY, indent=2))

## 9. Optional MTP-only domain fine-tuning
The target must generate its own responses; existing teacher responses are not automatically on-policy. Regeneration uses training images selected from with-tools trace files and excludes fixed dev membership. Image views and tool execution are retained. Multimodal render/hidden-state support is version-gated: if your pinned version rejects images, stop and report it—never drop image tokens. The official Speculators workflow is documented in docs/harness/SPECDEC.md. Separate environments avoid perturbing the existing training environment.

In [ ]:
RUN_MTP_FINETUNE = False
SPECULATORS_SOURCE = Path('data/speculators_source')  # A reviewed checkout of vllm-project/speculators matching your installed version.
TRAIN_TRACES = ['data/traces/train_cot_traces_dentex.jsonl','data/traces/train_cot_traces_healthy_dentex.jsonl']
TRAIN_EPISODES = 10  # Explicit smoke size; select the final training budget on train/dev, never test.
ON_POLICY = 'data/mtp_train/on_policy.jsonl'
MTP_OUTPUT = 'data/mtp_train/workflow'
MTP_EPOCHS = 3
MTP_SEQ_LENGTH = 16384
MTP_HIDDEN_DTYPE = 'float32' if HARDWARE_PROFILE.startswith('dual_t4') else 'bfloat16'  # Speculators does not support FP16 training autocast
MTP_ENDPOINT = f'http://127.0.0.1:{PORT}'
# Start the official hidden-state-capable vLLM wrapper in its serving environment.
# Source checkout is not downloaded silently; inspect/pin it before this opt-in stage.
HIDDEN_SERVER = None
if RUN_MTP_FINETUNE:
    if not MERGED: raise ValueError("Choose MTP_PROFILE and MTP_ADAPTER_PREFIX from SPEC_CHECKPOINTS before MTP training")
    if MODEL_ID is None:
        from dental_agent.utils.inputs import resolve_qwen_model
        MODEL_ID, _ = resolve_qwen_model()
    wrapper = SPECULATORS_SOURCE/'scripts/launch_vllm.py'
    assert wrapper.is_file(), 'Set SPECULATORS_SOURCE to a pinned reviewed Speculators checkout'
    sys.path.insert(0,str(Path.cwd()))
    from scripts.bench.spec_decode_bench import server
    serving_python = str((VLLM_ENV / ('Scripts/python.exe' if os.name=='nt' else 'bin/python')).resolve())
    hidden_cmd=[serving_python,str(wrapper.resolve()),str(Path(MERGED).resolve()),'--target-layer-ids','32','--',
                '--host','127.0.0.1','--port',str(PORT),'--max-model-len',str(MAX_MODEL_LEN),
                '--dtype',SERVING_DTYPE,'--tensor-parallel-size',str(TENSOR_PARALLEL),'--gpu-memory-utilization',str(GPU_MEMORY_UTILIZATION)]
    HIDDEN_SERVER=server(hidden_cmd,Path(MTP_OUTPUT)/'hidden_server.log',PORT,GPU_INDICES,600)
    HIDDEN_SERVER.__enter__()
    import atexit
    atexit.register(lambda: HIDDEN_SERVER.__exit__(None,None,None) if HIDDEN_SERVER is not None else None)

In [ ]:
if RUN_MTP_FINETUNE:
    subprocess.run([sys.executable,'scripts/bench/regenerate_train.py','--traces',*TRAIN_TRACES,
                    '--endpoint',MTP_ENDPOINT,'--data-dir',DATA_DIR,'--dev-split',DEV_SPLIT,'--output',ON_POLICY,
                    '--limit',str(TRAIN_EPISODES),'--max-new-tokens',str(MAX_NEW_TOKENS),
                    '--max-tool-calls',str(MAX_TOOL_CALLS),'--seed',str(SEED)],check=True)
    MTP_CMD=[sys.executable,'scripts/bench/mtp_finetune.py','--checkpoint',MERGED,'--conversations',ON_POLICY,
             '--output-dir',MTP_OUTPUT,'--endpoint',MTP_ENDPOINT,'--speculators-executable',SPECULATORS,
             '--seq-length',str(MTP_SEQ_LENGTH),'--hidden-states-dtype',MTP_HIDDEN_DTYPE,'--epochs',str(MTP_EPOCHS)]
    for stage in ['prepare','extract']:
        subprocess.run(MTP_CMD+['--stage',stage,'--execute'],check=True)
    # Free only this notebook's owned server before single-GPU offline training.
    HIDDEN_SERVER.__exit__(None,None,None);HIDDEN_SERVER=None
    for stage in ['train','stitch']:
        subprocess.run(MTP_CMD+['--stage',stage,'--execute'],check=True)


## 10. Acceptance before vs after MTP training
The stitch step checks that all non-MTP target tensor bytes are unchanged. Re-run the identical dev benchmark on the stitched checkpoint. Both runs must pass greedy equivalence, and dev exact F1 must remain unchanged; otherwise report the divergence.

In [ ]:
if RUN_MTP_FINETUNE:
    import copy
    AFTER_TARGETS = copy.deepcopy(SPEC_CHECKPOINTS)
    AFTER_TARGETS[MTP_PROFILE][MTP_ADAPTER_PREFIX] = str(Path(MTP_OUTPUT)/"stitched")
    after_evaluations = NotebookEvaluations(track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=MTP_PROFILE, kinds=["speculative"],
        stages=EVAL_STAGES, selected_k=EVAL_SELECTED_K, model_id=MODEL_ID,
        data_dir=DATA_DIR, dev_split=DEV_SPLIT, split="dev", spec_checkpoints=AFTER_TARGETS,
        vllm_executable=VLLM, gpu_indices=GPU_INDICES, port=PORT, serving_context_length=MAX_MODEL_LEN)
    AFTER_STATUS = after_evaluations.run()
    before = next(x for x in SPEC_EVAL_STATUS if x.get("adapter")==MTP_ADAPTER_PREFIX and x.get("path"))
    after = next(x for x in AFTER_STATUS if x.get("adapter")==MTP_ADAPTER_PREFIX and x.get("path"))
    subprocess.run([sys.executable,"scripts/bench/compare_acceptance.py","--before",str(Path(before["path"])/"results.json"),
        "--after",str(Path(after["path"])/"results.json"),"--output",str(Path(after["path"])/"acceptance_change.json")],check=True)
else:
    print("MTP training and acceptance comparison have not been run.")

## 11. Results on Hugging Face
Each benchmark is saved under the existing artifact repository. No paper build, automatic Git push or new repository creation.

In [ ]:
for item in SPEC_EVAL_STATUS:
    print(item.get("hub_path", item.get("reason", item["status"])))